In [1]:
import sys 
from pathlib import Path 
sys.path.append(str(Path().resolve().parents[0]))

import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

from config import PROCESSED_DATA_DIR, MODELS_DIR

In [2]:
pokemon_df = pd.read_parquet(PROCESSED_DATA_DIR / "pokemon_data_features.parquet")

In [3]:
y = pokemon_df["total_points"]
X = pokemon_df.drop(columns=["total_points"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [4]:
param_grid = {
    "n_estimators": [200, 300, 500],
    "learning_rate": [0.03, 0.05, 0.1],
    "max_depth": [3, 4, 5],
    "min_child_weight": [1, 3, 5],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
}

grid_search = GridSearchCV(
    estimator=XGBRegressor(objective="reg:squarederror", random_state=42),
    param_grid=param_grid,
    cv=3,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best CV RMSE: {-grid_search.best_score_:.2f}")

Fitting 3 folds for each of 324 candidates, totalling 972 fits
Best Parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 5, 'min_child_weight': 3, 'n_estimators': 200, 'subsample': 1.0}
Best CV RMSE: 66.46


In [5]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Tuned XGBoost RMSE: {rmse:.2f}")
print(f"Tuned XGBoost MAE: {mae:.2f}")
print(f"Tuned XGBoost R^2: {r2:.3f}")

Tuned XGBoost RMSE: 60.49
Tuned XGBoost MAE: 45.69
Tuned XGBoost R^2: 0.761


In [6]:
joblib.dump(best_model, MODELS_DIR / "xg_boost_initial.joblib")

['/Users/brandonwu/Documents/Pokemon_Project/models/xg_boost_initial.joblib']